# qlang

Qlang is designed as a very terse query language for data exploration, modification and formatting of pandas dataframes containing very messy data.

Note that some examples make use of colors which might not render depending on where this notebook is viewed (eg: github).

In [13]:
import pandas as pd
import numpy as np
import dukit as dk
from dukit import get_df, log, qs
pd.set_option('display.max_columns', None)

#this is our test dataset containing
#very messy, definitely real medical data
df = get_df()
df

,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
0,10001,John Doe,1995-01-02 00:00:00,-25,M,170,70.2,20,80,Normal,False,10kg
1,10002,Jane Smith,1990-09-14 00:00:00,30,F,175.5cm,68,130,85,Highe,true,NaN
2,10003,Alice Johnson,1985.08.23,NaN,Female,None,72.5lb,<NA>,N/A,NaN,N/A,15 mg once a day
3,20001,Bob Brown,19800406,NaT,Male,280,na,-inf,90mmHg,GOOD,0,20mg
4,20002,eva white,05-11-2007,40.0,Other,NULL,,135mmhg,NaN,n.a.,1,20 Mg
5,20003,Frank miller,1983-06-30,forty-five,m,185,75kg,125.5,75,High,Yes,25g
6,30001,Grace TAYLOR,1975-05-28,nan,ff,1,None,NAN,NaN,Normal,NO,NaN
7,30002,Harry Clark,1960Mar08,unk,NaN,6ft 1in,80.3,"122,3",<NA>,n/a,None,<NA>
8,30003,IVY GREEN,1955-Jan-09,,<NA>,-10,130lbs,,95,high,NaN,30 MG
9,30004,JAck Williams,1950 Sep 10,unknown,Mal,,1e2,130,0,,n,35


In [14]:
#qlang can be used as a standalone function:
qs(df, '')

#or via an accessor extension:
df.qs('')  #type: ignore (typecheckers do not like this)

#or via the dk accessor extension:
df.dk.qs('')

,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
0,10001,John Doe,1995-01-02 00:00:00,-25,M,170,70.2,20,80,Normal,False,10kg
1,10002,Jane Smith,1990-09-14 00:00:00,30,F,175.5cm,68,130,85,Highe,true,NaN
2,10003,Alice Johnson,1985.08.23,NaN,Female,None,72.5lb,<NA>,N/A,NaN,N/A,15 mg once a day
3,20001,Bob Brown,19800406,NaT,Male,280,na,-inf,90mmHg,GOOD,0,20mg
4,20002,eva white,05-11-2007,40.0,Other,NULL,,135mmhg,NaN,n.a.,1,20 Mg
5,20003,Frank miller,1983-06-30,forty-five,m,185,75kg,125.5,75,High,Yes,25g
6,30001,Grace TAYLOR,1975-05-28,nan,ff,1,None,NAN,NaN,Normal,NO,NaN
7,30002,Harry Clark,1960Mar08,unk,NaN,6ft 1in,80.3,"122,3",<NA>,n/a,None,<NA>
8,30003,IVY GREEN,1955-Jan-09,,<NA>,-10,130lbs,,95,high,NaN,30 MG
9,30004,JAck Williams,1950 Sep 10,unknown,Mal,,1e2,130,0,,n,35


## basics

### getters and setters

In [15]:
#get/select/filter cols by name 
df.dk.qs('age')

,age
0,-25
1,30
2,NaN
3,NaT
4,40.0
5,forty-five
6,nan
7,unk
8,
9,unknown


In [16]:
#get/select/filter rows using a getter operator
df.dk.qs('age  <0')

,age
0,-25


In [17]:
#set/change/modify the currently selected values using a setter operator
df.dk.qs('age  <0  =INVALID')

,age
0,INVALID


In [18]:
#rows where "john" is contained in the "name" col
df.dk.qs('name  ?john')

,name
0,John Doe
2,Alice Johnson
10,john Doe


In [19]:
#rows where the letter "y" is contained in any col
df.dk.qs('?y')

,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
2,10003,Alice Johnson,1985.08.23,NaN,Female,None,72.5lb,<NA>,N/A,NaN,N/A,15 mg once a day
5,20003,Frank miller,1983-06-30,forty-five,m,185,75kg,125.5,75,High,Yes,25g
6,30001,Grace TAYLOR,1975-05-28,nan,ff,1,None,NAN,NaN,Normal,NO,NaN
7,30002,Harry Clark,1960Mar08,unk,NaN,6ft 1in,80.3,"122,3",<NA>,n/a,None,<NA>
8,30003,IVY GREEN,1955-Jan-09,,<NA>,-10,130lbs,,95,high,NaN,30 MG


### named operators

unlike "<" or "=" some getter and setter operators dont have their own glyph, but are instead called by writing their name and a prefix.

the ":" prefix is used for getters, and the "." prefix is used for setters.

In [20]:
#set name to lowercase
df.dk.qs('name .lower()')
df.dk.qs('name .lower')  #brackets are optional but usefull when multiple args are used

,name
0,john doe
1,jane smith
2,alice johnson
3,bob brown
4,eva white
5,frank miller
6,grace taylor
7,harry clark
8,ivy green
9,jack williams


In [21]:
#reset selection by selecting all rows
df.dk.qs('age  <0  :all')

,age
0,-25
1,30
2,NaN
3,NaT
4,40.0
5,forty-five
6,nan
7,unk
8,
9,unknown


### scopes

while this minimal syntax is sometimes sufficient, 
more complex queries benefit from specifying
the scopes of operations explicitly.

the scopes are:
- '%':      cols/columns
- '%%':     rows/index
- '%%%':    vals/values/cells


In [22]:
#this previous query
df.dk.qs('age  <0  =invalid')

#is functinally the same as this more explicit version
df.dk.qs(r'%age  %%<0  %%%=invalid')


,age
0,invalid


as seen in this example:
- bare literals (names) default to col selection
- getters default to row selection
- setters default to value modification

explicit scopes can be used to override these defaults.

In [23]:
#get cols using the "contains" getter operator
df.dk.qs('%?bp')

,bp systole,bp diastole
0,20,80
1,130,85
2,<NA>,N/A
3,-inf,90mmHg
4,135mmhg,NaN
5,125.5,75
6,NAN,NaN
7,"122,3",<NA>
8,,95
9,130,0


In [24]:
#get specific values (without changing the row selection)
df.dk.qs(r'age  %%%<0  =INVALID')

,age
0,INVALID
1,30
2,NaN
3,NaT
4,40.0
5,forty-five
6,nan
7,unk
8,
9,unknown


In [25]:
#scopes without an operator default
#to selecting all values using ":all"
code = r"""
name  %%?j
%
%%
"""
df.dk.qs(code)

,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
0,10001,John Doe,1995-01-02 00:00:00,-25,M,170,70.2,20,80,Normal,False,10kg
1,10002,Jane Smith,1990-09-14 00:00:00,30,F,175.5cm,68,130,85,Highe,true,NaN
2,10003,Alice Johnson,1985.08.23,NaN,Female,None,72.5lb,<NA>,N/A,NaN,N/A,15 mg once a day
3,20001,Bob Brown,19800406,NaT,Male,280,na,-inf,90mmHg,GOOD,0,20mg
4,20002,eva white,05-11-2007,40.0,Other,NULL,,135mmhg,NaN,n.a.,1,20 Mg
5,20003,Frank miller,1983-06-30,forty-five,m,185,75kg,125.5,75,High,Yes,25g
6,30001,Grace TAYLOR,1975-05-28,nan,ff,1,None,NAN,NaN,Normal,NO,NaN
7,30002,Harry Clark,1960Mar08,unk,NaN,6ft 1in,80.3,"122,3",<NA>,n/a,None,<NA>
8,30003,IVY GREEN,1955-Jan-09,,<NA>,-10,130lbs,,95,high,NaN,30 MG
9,30004,JAck Williams,1950 Sep 10,unknown,Mal,,1e2,130,0,,n,35


In [26]:
#set cols/headers and rows/indices
code = r"""
name  %='full name'
age  <0  %%=99
"""
df.dk.qs(code)

,age
99,-25


### connector scopes

scopes are also used to connect results from multiple getter operations.

connector scopes:
- '&':    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;        cols in current selection AND next col getter selection
- '&&':   &nbsp;&nbsp;&nbsp;&nbsp;                          rows in current selection AND next row getter selection
- '&&&':  &nbsp;                                            vals in current selection AND next val getter selection
- '/':    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;  cols in current selection OR next col getter selection
- '//':   &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;        rows in current selection OR next row getter selection
- '///':  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;              vals in current selection OR next val getter selection

In [27]:
code = r"""
name
    %%?john
    &&?alice

/name
"""
df.dk.qs(code)

,name
2,Alice Johnson


In [28]:
#row getters only check their conditions
#in the currently selected cols but can be
#combined with previous row getters even
#if they were based on other cols

code = r"""
age
    %%<30
"date of birth"
    //>2000-01-01
%
"""
df.dk.qs(code)

,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
0,10001,John Doe,1995-01-02 00:00:00,-25,M,170,70.2,20,80,Normal,False,10kg
4,20002,eva white,05-11-2007,40.0,Other,NULL,,135mmhg,NaN,n.a.,1,20 Mg


### flags

A number of flags can be used to modify the behaviour of selection conditions.

The negation flag is used right in front of an operator, but most other flags are passed to the operator just like arguments, but with the "+" prefix to distinguish them from regular arguments.

In [29]:
#negate a condition
df.dk.qs(r'id   %%!>20000')

,ID
0,10001
1,10002
2,10003


In [30]:
#use the "+strict" flage to make
#selection criteria stricter.
#for the contains operator "?"
#this means case sensitivity.
df.dk.qs(r'name  ?john  +strict')

,name
10,john Doe


In [31]:
#can be negated as expected
df.dk.qs(r'name  !?john  +strict')

,name
0,John Doe
1,Jane Smith
2,Alice Johnson
3,Bob Brown
4,eva white
5,Frank miller
6,Grace TAYLOR
7,Harry Clark
8,IVY GREEN
9,JAck Williams


In [32]:
#normally, getters select rows where
#the condition is fulfilled in any
#of the currently selected cols.
df.dk.qs(r'height  /weight  :isnum +strict')

,height,weight
0,170,70.2
8,-10,130lbs
10,200,-65


In [33]:
#use the "+allcols" flag to select
#only rows where the condition is
#fulfilled in all selected cols.
df.dk.qs(r'height  /weight  :isnum +strict +allcols')

,height,weight
0,170,70.2
10,200,-65


### typing

The query language is designed to handle very messy datasets where sometimes no strict typing (or any typing at all!) is enforced during data entry. Therefor, operators like ":isdate" do not filter based on the types in the dataset (sometimes all values are strings), but rather if it makes sense for a value to be of a certain type. Obviously, what makes sense depends on the domain and the assumptions for qlang might not align with your use case.

Using the flag "strict" switches to stricter typing.

In [34]:
#display additional type info
#first is the unmodified value,
#second the inferred type in square brackets,
#third the converted result
df.dk.qs('age  /weight  .typeinfo  .wrap(pre-wrap)')

,age,weight
0,-25 [int] -25,70.2 [float] 70.2
1,'30' [int] 30,'68' [int] 68
2,nan [float] nan,'72.5lb' [str] '72.5lb'
3,NaT [na] None,'na' [na] None
4,'40.0' [float] 40.0,'' [na] None
5,'forty-five' [str] 'forty-five','75kg' [str] '75kg'
6,'nan' [na] None,None [na] None
7,'unk' [str] 'unk','80.3' [float] 80.3
8,'' [na] None,'130lbs' [str] '130lbs'
9,'unknown' [str] 'unknown','1e2' [num] np.float64(100.0)


In [35]:
#qlang infers that "40.0" should
#be a float, but due to the leniency
#requirement it is also recognized as a
#valid int if asked
df.dk.qs(r'age  :isint')

,age
0,-25
1,30
4,40.0
10,35


In [36]:
#avoiding leniency with the "+strict" flag
df.dk.qs(r'age  :isint +strict')

,age
0,-25
10,35


In [37]:
#type setters coerce values and
#fill with na where conversion fails
df.dk.qs(r'age  .copy("age int") .toint  /age')

,age,age int
0,-25,-25
1,30,30
2,NaN,<NA>
3,NaT,<NA>
4,40.0,40
5,forty-five,<NA>
6,nan,<NA>
7,unk,<NA>
8,,<NA>
9,unknown,<NA>


### getter logic

Due to expecting very messy data, qplib uses a type of [three-valued logic](https://en.wikipedia.org/wiki/Three-valued_logic) utilizing "True", "False" and "undefined". This means that, for example, numeric operators for numbers can be used on columns which also contain strings. As a result, ">=" is not necessarily the same as "!<" (inversion of "<").

In [38]:
df.dk.qs(r'height  %%>0')

,height
0,170
3,280
5,185
6,1
10,200


In [39]:
df.dk.qs(r'height  %%<0')

,height
8,-10


In [40]:
df.dk.qs(r'height  %%!>0')

,height
1,175.5cm
2,None
4,NULL
7,6ft 1in
8,-10
9,


In [41]:
#notice that only -10 is recognized as
#both <0 (orange background) and !>0 (red font)
#since the other values are not numbers,
#they can be !>0 but not <0
code = r"""
height
    %%>0   .bg(lime)
    %%<0   .bg(orange)
    %%!>0  .color(red)
    %%
"""
df.dk.qs(code)


,height
0,170
1,175.5cm
2,None
3,280
4,NULL
5,185
6,1
7,6ft 1in
8,-10
9,


In [42]:
#The same result but marking values via a second meta col
#in case the notebook renderer does not support highlighting:
df.dk.qs(
    r"""
    height  >0  .tag('>0 <br>')
    height  <0  .tag('<0 <br>')
    height  !>0  .tag('!>0 <br>')
    /_meta
    %%
    .align(right)
    """
    )

,height,_meta
0,170,>0
1,175.5cm,!>0
2,None,!>0
3,280,>0
4,NULL,!>0
5,185,>0
6,1,>0
7,6ft 1in,!>0
8,-10,<0 !>0
9,,!>0


### new cols

In [43]:
#new col is filled with pd.NA if only a name is provided
df.dk.qs('.new(a)')

,a
0,<NA>
1,<NA>
2,<NA>
3,<NA>
4,<NA>
5,<NA>
6,<NA>
7,<NA>
8,<NA>
9,<NA>


In [44]:
#fill new col with a value:
df.dk.qs('.new(a, 1)')

,a
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,1
8,1
9,1


In [45]:
#specify type:
df.dk.qs('.new(a, 1, +str)')

,a
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,1
8,1
9,1


## debugging

There are som helper operations executed during query execution starting with "." and some executed during query parsing starting with "..".


In [46]:
#show all symbols which are valid
#at the position of the  "..help" op
df.dk.qs(r'..help')

available scopes:


,name,lexeme,description,example
0,ScopeValsNew,%%%,"either: - set vals - get vals, replacing the previous val selection","qs(df, r'%%%:isint') qs(df, r'height /weight %%%:isint') #integer vals in height and weight cols"
1,ScopeValsAnd,&&&,get val selection fulfilling this AND the previous condition(s).,"qs(df, r'%%%:isint &&&>0') #integer vals above 0"
2,ScopeValsOr,///,get val selection fulfilling this OR the previous condition(s).,"qs(df, r'%%%:isint ///:isna') #integer or na vals"
3,ScopeRowsNew,%%,"either: - set rows - get rows, replacing the previous row selection","qs(df, r'age %%>30')"
4,ScopeRowsAnd,&&,get row selection fulfilling this AND the previous condition(s).,"qs(df, r'age %%>30 &&<50') #rows where age >30 and <50"
5,ScopeRowsOr,//,get row selection fulfilling this OR the previous condition(s).,"qs(df, r'name %%?a //?b') #rows where name contains ""a"" OR ""b"""
6,ScopeColsNew,%,"either: - set cols - get cols, replacing the previous col selection","qs(df, r'%==age')"
7,ScopeColsAnd,&,get col selection fulfilling this AND the previous condition(s).,"qs(df, r'%?a &?e') #cols containing ""a"" AND ""e"""
8,ScopeColsOr,/,get col selection fulfilling this OR the previous condition(s).,"qs(df, r'%?a /?e') #cols containing ""a"" OR ""e"""


get/select/filter cols/rows/vals:


,name,lexeme,description,example
0,GetEquals,==,get cols/rows/vals if they are equal to an arg.,"qs(df, r'age ==30')"
1,GetNotEquals,!=,get cols/rows/vals if they are not equal to an arg.,"qs(df, r'age !=30')"
2,GetContains,?,get cols/rows/vals if they contain an arg.,"qs(df, r'name ?john')"
3,GetGreaterEqual,>=,get cols/rows/vals if they are greater than or equal to an arg.,"qs(df, r'age >=30')"
4,GetSmallerEqual,<=,get cols/rows/vals if they are smaller than or equal to an arg.,"qs(df, r'age <=30')"
5,GetGreater,>,get cols/rows/vals if they are greater than an arg.,"qs(df, r'age >30')"
6,GetSmaller,<,get cols/rows/vals if they are smaller than an arg.,"qs(df, r'age <30')"
7,GetEval,:apply :eval :map,"get all cols/rows/vals where a custom python expression evaluates to True. ""x"" can be used in the expression to refer to the current item.","qs(df, r'%:eval(""len(x) > 3"")') #cols with names longer than 3 characters"
8,GetSavedSelection,:load,load a previously saved selection for the current scope.,"qs(df, r'name .save(1) %age %:load(1)')"
9,GetTrimmedSelection,:trim,trim the current row or col selection to entries which contain currently selected vals.,"qs(df, r'%%%>0 &&&<100 %:trim') qs(df, r'%%%>0 &&&<100 %%:trim')"


set/change/modify cols/rows/vals:


...

change style of cols/rows/vals:


,name,lexeme,description,example
0,StyleMonospace,.mono-space .mono,change the font family of selected cols/rows/vals to monospaced consolas font,"qs(df, r'%.mono')"
1,StyleFont,.font-weight .font-style .fontweight .fontstyle .weight .style .font,change the font style of selected cols/rows/vals,"qs(df, r'%.font(bold)')"
2,StyleColor,.colour .color,change the text color of selected cols/rows/vals,"qs(df, r'name .color(red)')"
3,StyleBackgroundColor,.background_colour .background_color .backgroundcolour .backgroundcolor .background .bg,change the background color of selected cols/rows/vals,"qs(df, r'name .bg(red)')"
4,StyleAlignement,.alignment .align,change the alignment of selected cols/rows/vals,"qs(df, r'name .align(center)')"
5,StyleTextWrap,.whitespace .text-wrap .textwrap .wrap,change the text wrapping style of selected cols/rows/vals,"qs(df, r'name .wrap(nowrap)')"


view debug information:


,name,lexeme,description,example
0,View,.display .print .show .view,view the currently selected cols/rows/vals. use when debugging long queries. can also print an optional arg.,"qs(df, r'%%%:isna .view(step1) %name ?john .view')"
1,ViewQuery,.view_query .query .q,view the query object itself. use when debugging long queries. can also print an optional arg.,"qs(df, r'%%%:isna .q(step1) %name ?john .q')"
2,ViewMasks,.view_masks .masks,view the current selection masks. use when debugging long queries. can also print an optional arg.,"qs(df, r'%%%:isna .masks(step1) %name ?john .masks')"
3,ViewMaskCols,.view_mask_cols .mask_cols .cols,view the current col selection masks. use when debugging long queries. can also print an optional arg.,"qs(df, r'%%%:isna .cols(step1) %name ?john .cols')"
4,ViewMaskRows,.view_mask_rows .mask_rows .rows,view the current row selection masks. use when debugging long queries. can also print an optional arg.,"qs(df, r'%%%:isna .rows(step1) %name ?john .rows')"
5,ViewMaskVals,.view_mask_vals .mask_vals .vals,print the current val selection masks. use when debugging long queries. can also print an optional arg.,"qs(df, r'%%%:isna .vals(step1) %name ?john .vals')"
6,ViewSetNArep,.na_rep .narep,specify how NA values are displayed,"qs(df, r'name .na_rep(""N/A"")')"
7,ViewParserHelp,..help,view debug information about currently available symbols during parsing,"qs(df, r'..help')"
8,ViewParserQuery,..view_query ..query ..q,view debug information about the Query object during parsing,"qs(df, r'age ..q')"
9,ViewParserOps,..ops,view debug information about current ops during parsing,"qs(df, r'age ..ops')"


,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
0,10001,John Doe,1995-01-02 00:00:00,-25,M,170,70.2,20,80,Normal,False,10kg
1,10002,Jane Smith,1990-09-14 00:00:00,30,F,175.5cm,68,130,85,Highe,true,NaN
2,10003,Alice Johnson,1985.08.23,NaN,Female,None,72.5lb,<NA>,N/A,NaN,N/A,15 mg once a day
3,20001,Bob Brown,19800406,NaT,Male,280,na,-inf,90mmHg,GOOD,0,20mg
4,20002,eva white,05-11-2007,40.0,Other,NULL,,135mmhg,NaN,n.a.,1,20 Mg
5,20003,Frank miller,1983-06-30,forty-five,m,185,75kg,125.5,75,High,Yes,25g
6,30001,Grace TAYLOR,1975-05-28,nan,ff,1,None,NAN,NaN,Normal,NO,NaN
7,30002,Harry Clark,1960Mar08,unk,NaN,6ft 1in,80.3,"122,3",<NA>,n/a,None,<NA>
8,30003,IVY GREEN,1955-Jan-09,,<NA>,-10,130lbs,,95,high,NaN,30 MG
9,30004,JAck Williams,1950 Sep 10,unknown,Mal,,1e2,130,0,,n,35


## logging


In [47]:
#on the default verbosity level 3,
#qlang shows errors for invalid queries
#and warnings for queries that
#might not yield the desired results.
df.dk.qs('fullname ?john')

1,WARNING,no cols fulfill the condition in current op.,function: _get_colsselected_cols: 12selected_rows: 11selected_vals: 132op:Operation 0: connector: new scope: cols operator: GetEquals args: ['fullname'] flags: {},2026-07-10 14:52:41.740618,0.000000,0.000000


2,ERROR,row selection cannot be applied when the current col selection is empty.,function: _get_rowsselected_cols: 0selected_rows: 11selected_vals: 0op:Operation 1: connector: new scope: rows operator: GetContains args: ['john'] flags: {},2026-07-10 14:52:41.756947,16.338000,16.340000


""
0
1
2
3
4
5
6
7
8
9


In [48]:
#verbosity 5 will show more information
df.dk.qs('fullname ?john', verbosity=5)

3,DEBUG,scanned code into 6 tokens.,function: scancode: fullname ?johnlines: 1tokens: QueryStart Literal Whitespace GetContains Literal QueryStop,2026-07-10 14:52:41.792416,51.811000,35.485000


4,TRACE,inferring cols scope for literal token.,function: _preparse_for_literal,2026-07-10 14:52:41.805980,65.370000,13.574000


5,TRACE,saving valid op.,function: _process_opop:Operation current: connector: new scope: cols operator: GetEquals args: ['fullname'] flags: {},2026-07-10 14:52:41.818057,77.450000,12.092000


6,TRACE,inferring rows scope for getter.,function: _preparse_for_getter,2026-07-10 14:52:41.828966,88.356000,10.919000


7,TRACE,saving valid op.,function: _process_opop:Operation current: connector: new scope: rows operator: GetContains args: ['john'] flags: {},2026-07-10 14:52:41.841229,100.618000,12.272000


8,DEBUG,parsed tokens into 2 ops.,function: parseops: GetEquals GetContains,2026-07-10 14:52:41.856286,115.676000,15.067000


9,WARNING,no cols fulfill the condition in current op.,function: _get_colsselected_cols: 12selected_rows: 11selected_vals: 132op:Operation 0: connector: new scope: cols operator: GetEquals args: ['fullname'] flags: {},2026-07-10 14:52:41.872338,131.729000,16.063000


10,ERROR,row selection cannot be applied when the current col selection is empty.,function: _get_rowsselected_cols: 0selected_rows: 11selected_vals: 0op:Operation 1: connector: new scope: rows operator: GetContains args: ['john'] flags: {},2026-07-10 14:52:41.887373,146.766000,15.050000


11,DEBUG,ran 2 ops.,function: run,2026-07-10 14:52:41.899806,159.196000,12.444000


""
0
1
2
3
4
5
6
7
8
9


In [49]:
#all logs from the current session
#(since importing dukit) can be found here:
log()

,level,text,context,time,total_ms,delta_ms
0,WARNING,no cols fulfill the condition in current op.,function: _get_colsselected_cols: 12selected_rows: 11selected_vals: 132op:Operation 0: connector: new scope: cols operator: GetEquals args: ['fullname'] flags: {},2026-07-10 14:52:41.740618,0.000000,0.000000
1,ERROR,row selection cannot be applied when the current col selection is empty.,function: _get_rowsselected_cols: 0selected_rows: 11selected_vals: 0op:Operation 1: connector: new scope: rows operator: GetContains args: ['john'] flags: {},2026-07-10 14:52:41.756947,16.338000,16.340000
2,DEBUG,scanned code into 6 tokens.,function: scancode: fullname ?johnlines: 1tokens: QueryStart Literal Whitespace GetContains Literal QueryStop,2026-07-10 14:52:41.792416,51.811000,35.485000
3,TRACE,inferring cols scope for literal token.,function: _preparse_for_literal,2026-07-10 14:52:41.805980,65.370000,13.574000
4,TRACE,saving valid op.,function: _process_opop:Operation current: connector: new scope: cols operator: GetEquals args: ['fullname'] flags: {},2026-07-10 14:52:41.818057,77.450000,12.092000
5,TRACE,inferring rows scope for getter.,function: _preparse_for_getter,2026-07-10 14:52:41.828966,88.356000,10.919000
6,TRACE,saving valid op.,function: _process_opop:Operation current: connector: new scope: rows operator: GetContains args: ['john'] flags: {},2026-07-10 14:52:41.841229,100.618000,12.272000
7,DEBUG,parsed tokens into 2 ops.,function: parseops: GetEquals GetContains,2026-07-10 14:52:41.856286,115.676000,15.067000
8,WARNING,no cols fulfill the condition in current op.,function: _get_colsselected_cols: 12selected_rows: 11selected_vals: 132op:Operation 0: connector: new scope: cols operator: GetEquals args: ['fullname'] flags: {},2026-07-10 14:52:41.872338,131.729000,16.063000
9,ERROR,row selection cannot be applied when the current col selection is empty.,function: _get_rowsselected_cols: 0selected_rows: 11selected_vals: 0op:Operation 1: connector: new scope: rows operator: GetContains args: ['john'] flags: {},2026-07-10 14:52:41.887373,146.766000,15.050000


In [50]:
#the logs are stored in a styled dataframe,
#and therefor qlang can be used to filter them:
logs = log().data.copy()
qs(logs, r'level  %%debug  %:all .align(left)')

,level,text,context,time,total_ms,delta_ms
2,DEBUG,scanned code into 6 tokens.,function: scancode: fullname ?johnlines: 1tokens: QueryStart Literal Whitespace GetContains Literal QueryStop,2026-07-10 14:52:41.792416,51.811000,35.485000
7,DEBUG,parsed tokens into 2 ops.,function: parseops: GetEquals GetContains,2026-07-10 14:52:41.856286,115.676000,15.067000
10,DEBUG,ran 2 ops.,function: run,2026-07-10 14:52:41.899806,159.196000,12.444000


In [51]:
#clear logs:
log(clear=True)
logs = log().data.copy()
logs

cleared all logs in dukit.util.logs.


""


In [52]:
#print intermediate results during query execution
qs(
    df, 
    r"""
    id
        %%>20000
        &&<30000
    .show

    name
        %%?bob
    .show("selection at second show() call:")

    %
    %%
    """
    )

,ID
3,20001
4,20002
5,20003


selection at second show() call:


,name
3,Bob Brown


,ID,name,date of birth,age,gender,height,weight,bp systole,bp diastole,cholesterol,diabetes,dose
0,10001,John Doe,1995-01-02 00:00:00,-25,M,170,70.2,20,80,Normal,False,10kg
1,10002,Jane Smith,1990-09-14 00:00:00,30,F,175.5cm,68,130,85,Highe,true,NaN
2,10003,Alice Johnson,1985.08.23,NaN,Female,None,72.5lb,<NA>,N/A,NaN,N/A,15 mg once a day
3,20001,Bob Brown,19800406,NaT,Male,280,na,-inf,90mmHg,GOOD,0,20mg
4,20002,eva white,05-11-2007,40.0,Other,NULL,,135mmhg,NaN,n.a.,1,20 Mg
5,20003,Frank miller,1983-06-30,forty-five,m,185,75kg,125.5,75,High,Yes,25g
6,30001,Grace TAYLOR,1975-05-28,nan,ff,1,None,NAN,NaN,Normal,NO,NaN
7,30002,Harry Clark,1960Mar08,unk,NaN,6ft 1in,80.3,"122,3",<NA>,n/a,None,<NA>
8,30003,IVY GREEN,1955-Jan-09,,<NA>,-10,130lbs,,95,high,NaN,30 MG
9,30004,JAck Williams,1950 Sep 10,unknown,Mal,,1e2,130,0,,n,35


## more examples

In [53]:
#apply row getter condition to the index
df.dk.qs(r'weight  /height  >5 +index')

,height,weight
6,1,None
7,6ft 1in,80.3
8,-10,130lbs
9,,1e2
10,200,-65


In [54]:
#interpret the value for comparison as a regex:
df.dk.qs(r'name    ? "J...\s" +regex')

,name
0,John Doe
1,Jane Smith
9,JAck Williams


In [55]:
#combining data selection, modification, visualization

code = r"""


#format names

name
    .copy("first name")
    .eval("x.split()[0].capitalize()")  #eval custom python code on the selected values

name
    .copy("last name")
    .eval("x.split()[1].capitalize()")



#highlight invalid data

height /weight
    %.color(green)  #paint headers
    %%%!:isfloat  .bg(orange)  #select and highlight invalid values
    %%:trim  %%.color(orange)  #keep only rows were something was selected and highlight them
    %%

height
    #select valid vals
    %%%>140
    &&&<210
    .save(1)  #save with name "1"

weight
    %%%>30
    &&&<200
    .save(2)

"date of birth"
    %%%:all  .todate
    &&&>1950-01-01
    &&&<2000-01-01
    .save(3)

#combine saved valid values and highlight the inverse
height /weight /"date of birth"
    %%%:load(1)
    ///:load(2)
    ///:load(3)
    %%%:invert
    .color(red)


#reset selection
%
%%
%%%



"date of birth"
/height
/weight
/"first name"
/"last name"
    %.upper .align(center)
"""
df.dk.qs(code)

,DATE OF BIRTH,HEIGHT,WEIGHT,FIRST NAME,LAST NAME
0,1995-01-02,170,70.200000,John,Doe
1,1990-09-14,175.5cm,68,Jane,Smith
2,1985-08-23,None,72.5lb,Alice,Johnson
3,1980-04-06,280,na,Bob,Brown
4,2007-11-05,NULL,,Eva,White
5,1983-06-30,185,75kg,Frank,Miller
6,1975-05-28,1,None,Grace,Taylor
7,1960-03-08,6ft 1in,80.3,Harry,Clark
8,1955-01-09,-10,130lbs,Ivy,Green
9,1950-09-10,,1e2,Jack,Williams


In [56]:
#todo


# #values can also be replaced with
# #respective values from other cols:
# df.dk.qs(r'name  %%%?j  $vals=@ID  %is any;')

In [57]:
#todo


# #Or appended with values from other cols:
# df.dk.qs(
#     r"""
#     #create col with error codes
#     $new=___ERROR  $cols=error code
#         %%%is any;  $vals+=@ID


#     #append error codes to na values in subset of df
#     %age  /gender
#         %%idx>5  &&idx<=8
#             %%%is na;
#                 $vals+=@error code
#                 $bg=orange


#     is any;  %%is any;
#     """
#     )

# df reshaping

these functions offer various ways to deal with non-unique keys/ids.  
primarily used when merging dfs with a one-to-many relationship.

In [58]:
import dukit as dk

df1, df2 = dk.get_dfs()

display(df1, df2)

,id,name,age
0,10001,John Doe,25
1,10002,Jane Smith,30
2,20001,Alice Johnson,35
3,30001,Bob Brown,40


,id,medication,dose,unit
0,10001,Aspirin,100,mg
1,10001,Ibuprofen,200,mg
2,20001,Paracetamol,<NA>,
3,20001,Amoxicillin,250,mg
4,20001,Ciprofloxacin,500,ml
5,30001,Metformin,1000,mg


In [59]:
df1.merge(
    df2.dk.flatten(),
    on='id',
    )

,id,name,age,medication1,medication2,medication3,dose1,dose2,dose3,unit1,unit2,unit3
0,10001,John Doe,25,Aspirin,Ibuprofen,NaN,100,200.0,NaN,mg,mg,NaN
1,20001,Alice Johnson,35,Paracetamol,Amoxicillin,Ciprofloxacin,<NA>,250.0,500.0,,mg,ml
2,30001,Bob Brown,40,Metformin,NaN,NaN,1000,NaN,NaN,mg,NaN,NaN


In [60]:
df1.merge(
    df2.dk.stagger(),
    on='id',
    )

,id,name,age,medication1,dose1,unit1,medication2,dose2,unit2,medication3,dose3,unit3
0,10001,John Doe,25,Aspirin,100,mg,Ibuprofen,200.0,mg,NaN,NaN,NaN
1,20001,Alice Johnson,35,Paracetamol,<NA>,,Amoxicillin,250.0,mg,Ciprofloxacin,500.0,ml
2,30001,Bob Brown,40,Metformin,1000,mg,NaN,NaN,NaN,NaN,NaN,NaN


In [61]:
df1.merge(
    df2.dk.embed(),
    on='id',
    ).dk.style()

,id,name,age,1,2,3
0,10001,John Doe,25,medication: Aspirindose: 100unit: mg,medication: Ibuprofendose: 200unit: mg,
1,20001,Alice Johnson,35,medication: Paracetamoldose: unit:,medication: Amoxicillindose: 250unit: mg,medication: Ciprofloxacindose: 500unit: ml
2,30001,Bob Brown,40,medication: Metformindose: 1000unit: mg,,


In [62]:
df1.merge(
    df2.dk.collapse(),
    on='id',
    ).dk.style()

,id,name,age,medication,dose,unit
0,10001,John Doe,25,1: Aspirin2: Ibuprofen,1: 1002: 200,1: mg2: mg
1,20001,Alice Johnson,35,1: Paracetamol2: Amoxicillin3: Ciprofloxacin,1: 2: 2503: 500,1: 2: mg3: ml
2,30001,Bob Brown,40,Metformin,1000,mg
